# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** The learned model that has to beat my ML-07 rule —
same data, same metric, same split, errors read before the score is believed.

Continues from `w03_data_contract.ipynb` (which fields I may touch),
`w03_feature_leakage_check.ipynb` (which four are forbidden, and why one of them looks harmless) and
`w04_baseline_score.ipynb` (the rule scoring **precision@50 = 0.740** in-sample, and its two named
defects). ML-07 ended with a promise: *"ML-08 must compare model vs baseline on a client-held-out split,
scoring both on the same held-out clients."* This notebook keeps it.

**What I ship at the end:** not the strongest model, and not the rule — the **combination**. The rule
picks the band and supplies the reason codes; a logistic regression orders pages *inside* it. That fixes
the exact defect ML-07 diagnosed, and it is the only candidate whose precision falls monotonically as K
grows, which is what a correctly-ordered queue looks like.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import json
import numpy as np
import pandas as pd
import sklearn
from pandas.api.types import is_numeric_dtype

RANDOM_STATE = 42          # fixed everywhere: split, every model, the bootstrap
np.random.seed(RANDOM_STATE)

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is a -20% threshold on the 30-day impression pair.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


def roc_auc(labels, scores) -> float:
    """AUC via the rank-sum identity - same helper as ML-07, so the numbers are comparable."""
    labels = np.asarray(labels)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:            # a single-class slice has no AUC; say so rather than divide
        return float("nan")
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().to_numpy()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}  ({y.sum():,} of {len(y):,} pages measured as declining)")
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

Working dir: C:\Users\real time\Desktop\Rayan_flyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421  (16,262 of 30,000 pages measured as declining)
pandas 3.0.1 | numpy 2.4.0 | scikit-learn 1.9.0


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**The question is "which pages first?", so the method must produce a score, not a verdict.** Editorial
capacity is 50 slots against 30,000 pages — 0.17% of the portfolio. Every candidate below is a
classifier, but I never threshold it at 0.5 and report accuracy; I take `predict_proba` and evaluate the
**ordering** with precision@K, exactly as ML-07 evaluated the rule. That is the toolkit's *"which first?
→ any classifier's probability, read at precision@K"* row. Recall is close to meaningless here: with
16,262 positives and 50 slots, the ceiling is **0.31%**.

**Four candidates, in deliberate order of readability:**

| Model | Why it is in the comparison |
|---|---|
| **Logistic regression** | The readable floor. Scaled inputs, coefficients I can inspect. If this wins, complexity has not earned its place. |
| **Decision tree, depth 3** | Printable in ten lines — the model you can read out loud to a strategist. |
| **Random forest** | The reference pipeline's choice, so it is the familiar bar (`outputs/model_report.md`). |
| **Histogram gradient boosting** | The stronger tabular learner, included so that "the simple model won" cannot be dismissed as me under-trying the hard one. |

**Features come from the contract, not from convenience.** The 32 fields bucketed as `feature` in
`w03_data_contract.ipynb`, plus six `has_*` missing-indicator flags, with `avg_position == 0` recoded to
missing *first* — 0 means "no reading", not "rank zero", and 1,205 rows carry it. Missingness tracks
`content_type` here (feedly articles are 100% missing `search_volume`), so those flags are what stop a
median fill from smuggling "this is a feedly article" into the model dressed up as a demand signal.

**The heavy-tailed counts are `log1p`'d, as ML-05 specified** (skew 11–18). Trees are invariant to a
monotone transform, so this changes nothing for three of the four candidates. **I do it anyway so the
logistic row is a fair comparison and not a straw man** — and that decision turned out to matter more
than any other in this notebook: without it the linear model finishes last, with it, it finishes first.

**The four forbidden fields stay out, asserted rather than claimed:** `trend_direction`, `trend_pct`,
`impressions_last_30d`, `impressions_prev_30d`. ML-05 measured that the last two reconstruct the label at
**1.0000**, while `impressions_last_30d` alone looks like the *least* informative column in the file
(AUC 0.486) — the trap a univariate screen walks straight into.

**One honesty note on hyperparameters.** I ran no search. These are library defaults plus two
conservative variance guards (`min_samples_leaf`, a shallow `max_depth`). A tuned model would need its
own inner split, and an untuned model that wins is a cleaner claim than a tuned one that wins by tuning.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.base import clone

CAPACITY_PER_SPRINT = 50
print(f"portfolio {len(df):,} pages | review slots {CAPACITY_PER_SPRINT} "
      f"({CAPACITY_PER_SPRINT / len(df):.2%}) -> ranking, evaluated at precision@K")
print(f"max achievable recall at K=50: {CAPACITY_PER_SPRINT / y.sum():.2%} -> recall is not the metric\n")

# --- the feature matrix, straight from the ML-04 contract -------------------
FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]
X = df[FEATURES].copy()
X["avg_position"] = X["avg_position"].replace(0, np.nan)   # 0 == "no reading", not rank 0 (1,205 rows)

FLAG_COLS = ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position"]
for c in FLAG_COLS:
    X[f"has_{c}"] = X[c].notna().astype(int)

# log1p the heavy tails (ML-05 measured skew 11-18). Trees do not care; the linear model does, and
# doing it for everyone is what makes the logistic row a fair comparison rather than a straw man.
COUNT_COLS = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
              "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]
print("skew before log1p:", {c: round(float(df[c].skew()), 1) for c in COUNT_COLS[:4]})
X[COUNT_COLS] = np.log1p(X[COUNT_COLS])

CATEGORICAL = [c for c in FEATURES if not is_numeric_dtype(df[c])]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]
print(f"feature matrix: {X.shape[1]} columns = {len(NUMERIC)} numeric (incl. {len(FLAG_COLS)} has_* flags) "
      f"+ {len(CATEGORICAL)} categorical, one-hot encoded downstream")
print(f"categorical: {CATEGORICAL}\n")

# --- LEAKAGE GUARD: asserted, not claimed ----------------------------------
FORBIDDEN = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "is_declining_label", "clicks_last_30d", "clicks_prev_30d",
             "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"}
assert not FORBIDDEN & set(X.columns), f"forbidden column in the matrix: {FORBIDDEN & set(X.columns)}"
print(f"forbidden columns in the feature matrix: {sorted(FORBIDDEN & set(X.columns)) or 'none'}")
print("  (ML-05: impressions_last_30d + impressions_prev_30d reconstruct the label at 1.0000)\n")


def make_pre(scale: bool) -> ColumnTransformer:
    """Median-fill numerics (the has_* flags carry the missingness); explicit level for missing categories."""
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="__missing__")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CATEGORICAL),
    ])


MODELS = {
    "logistic_regression": (True,  LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)),
    "decision_tree_d3":    (False, DecisionTreeClassifier(max_depth=3, min_samples_leaf=200,
                                                          random_state=RANDOM_STATE)),
    "random_forest":       (False, RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                                          n_jobs=-1, random_state=RANDOM_STATE)),
    "hist_gradient_boost": (False, HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06,
                                                                  max_iter=300, random_state=RANDOM_STATE)),
}


def make_pipe(name: str) -> Pipeline:
    """clone() so no estimator state is ever shared between folds - a silent, ugly bug otherwise."""
    scale, est = MODELS[name]
    return Pipeline([("pre", make_pre(scale)), ("clf", clone(est))])


print("candidates:", list(MODELS))

# --- the ML-07 rule, rebuilt here so both systems score identical rows ------
position = df["avg_position"].replace(0, np.nan)
RULE = {
    "established_coverage": (df["days_with_impressions"].between(20, 87), 3),
    "has_demand":           (df["impressions_90d"] >= 40, 2),
    "mid_position":         (((position > 3) & (position <= 50)).fillna(False), 2),
    "stale_90d":            (df["days_since_last_update"] >= 90, 1),
    "mature_page":          (df["content_age_days"].between(90, 364), 1),
}
baseline_score = sum(flag.astype(int) * pts for flag, pts in RULE.values()).to_numpy()
baseline_rank_key = baseline_score * 100 + np.minimum(np.log1p(df["impressions_90d"]), 10).to_numpy()

committed = json.loads(open("work/outputs/baseline_metrics.json").read())
here = round(precision_at_k(y, baseline_rank_key, 50), 4)
print(f"\nbaseline rebuilt: precision@50 = {here} | committed ML-07 receipt = {committed['precision_at_50']}")
assert here == committed["precision_at_50"], "the rebuilt baseline does not match the ML-07 receipt"
print("  -> matches the committed receipt, so the comparison below is against the real ML-07 rule")

portfolio 30,000 pages | review slots 50 (0.17%) -> ranking, evaluated at precision@K
max achievable recall at K=50: 0.31% -> recall is not the metric

skew before log1p: {'impressions_90d': 11.4, 'clicks_90d': 18.3, 'pageviews_90d': 10.9, 'sessions_90d': 12.1}
feature matrix: 38 columns = 29 numeric (incl. 6 has_* flags) + 9 categorical, one-hot encoded downstream
categorical: ['competition_level', 'word_count_tier', 'char_count_tier', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'impression_tier', 'position_tier']

forbidden columns in the feature matrix: none
  (ML-05: impressions_last_30d + impressions_prev_30d reconstruct the label at 1.0000)

candidates: ['logistic_regression', 'decision_tree_d3', 'random_forest', 'hist_gradient_boost']

baseline rebuilt: precision@50 = 0.74 | committed ML-07 receipt = 0.74
  -> matches the committed receipt, so the comparison below is against the real ML-07 rule


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client. Not time-aware, because this file cannot be.**

**Why not time-aware.** ML-04 established that the starter file carries **no date column at all** — every
time field is a relative offset from an unstated snapshot. There is no past and no future in it, so a
time-aware split is not something I am declining to do, it is something the data forbids. A genuine
past→future split needs `fact_content_daily_performance` from the warehouse release (`report_date`
2025-01-27 → 2026-06-30). Stated as a limit, not worked around.

**Why grouped by client, in one measured number.** The panel is 32 clients holding 3 to 7,008 pages, with
per-client label rates from 0.000 to 0.937 (ML-04, limit 6). A random row split puts pages from the same
client on both sides, and the model learns the client's label rate instead of the decay signal. Measured
below with the same model and the same features:

| Split | ROC-AUC | precision@50 |
|---|---|---|
| Random 80/20 row split | **0.777** | 0.920 |
| Grouped, clients held out | **0.624** | 0.780 |

**That 0.152 AUC gap is client memorisation, not skill.** If I reported the random-split number, most of
my headline would be the model recognising which client a page belongs to — worthless for the actual
decision, because a new client arrives with no history at all. Every number in Section 3 uses the grouped
split. This is also the single easiest way to accidentally publish an inflated result on this dataset,
which is why it is measured here rather than asserted.

**GroupKFold with 5 folds rather than one held-out set,** for a specific reason: with 32 clients and one
of them holding 23% of all rows, a single 80/20 client split is a lottery. Cross-validation lets me
report both a pooled estimate and the **spread across folds** — and in Section 3 the spread turns out to
be the more interesting number.

**The pooled out-of-fold ranking is the apples-to-apples object.** Each page is scored by the one
fold-model that never saw its client; the 30,000 predictions are then ranked as a single queue. That
mirrors the real use — one sprint queue over the whole portfolio — and it means both systems rank
identical rows.

**One asymmetry I have to disclose, because it runs against my own result.** The ML-07 rule's thresholds
were read off crosstabs computed on all 30,000 rows, so the rule is mildly in-sample everywhere, while
every model number is strictly out-of-fold. **The comparison is tilted in the baseline's favour, not the
model's.** I am reporting it that way rather than re-fitting the rule per fold, because the rule as
submitted for ML-07 is the thing that has to be beaten.

**Fold 1 is a single client.** GroupKFold packs the 7,008-page client into a fold of its own — visible in
the table below. Its numbers are a one-client test and I read them as such.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold, train_test_split

gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, groups))

print("Fold composition (clients are held out whole):")
print(f"  {'fold':>4} {'clients':>8} {'rows':>8} {'label rate':>11}")
for i, (tr, te) in enumerate(folds):
    print(f"  {i+1:>4} {len(np.unique(groups[te])):>8} {len(te):>8,} {y[te].mean():>11.3f}")
print(f"  -> fold label rates span {min(y[te].mean() for _, te in folds):.3f} to "
      f"{max(y[te].mean() for _, te in folds):.3f}: clients are not interchangeable")

for i, (tr, te) in enumerate(folds):
    assert not (set(groups[tr]) & set(groups[te])), f"client leaked across fold {i+1}"
print("  -> asserted: zero client overlap between train and test in all 5 folds\n")

# --- the number that justifies the split -----------------------------------
PROBE = "hist_gradient_boost"
tr_r, te_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
p_rand = make_pipe(PROBE).fit(X.iloc[tr_r], y[tr_r]).predict_proba(X.iloc[te_r])[:, 1]

tr_g, te_g = folds[1]                      # a multi-client fold, not the single-client fold 1
p_grp = make_pipe(PROBE).fit(X.iloc[tr_g], y[tr_g]).predict_proba(X.iloc[te_g])[:, 1]

print(f"Same model ({PROBE}), same features, two splits:")
print(f"  random 80/20 row split    AUC {roc_auc(y[te_r], p_rand):.4f}   p@50 {precision_at_k(y[te_r], p_rand, 50):.3f}"
      f"   ({len(te_r):,} test rows)")
print(f"  grouped, clients held out AUC {roc_auc(y[te_g], p_grp):.4f}   p@50 {precision_at_k(y[te_g], p_grp, 50):.3f}"
      f"   ({len(te_g):,} test rows)")
print(f"  inflation from letting clients cross the split: "
      f"{roc_auc(y[te_r], p_rand) - roc_auc(y[te_g], p_grp):+.4f} AUC")
print("  -> that gap is client memorisation. Every number below uses the grouped split.")

Fold composition (clients are held out whole):
  fold  clients     rows  label rate
     1        1    7,008       0.490
     2        7    5,731       0.645
     3        8    5,753       0.379


     4        8    5,755       0.622
     5        8    5,753       0.585
  -> fold label rates span 0.379 to 0.645: clients are not interchangeable
  -> asserted: zero client overlap between train and test in all 5 folds



Same model (hist_gradient_boost), same features, two splits:
  random 80/20 row split    AUC 0.7765   p@50 0.920   (6,000 test rows)
  grouped, clients held out AUC 0.6244   p@50 0.780   (5,731 test rows)
  inflation from letting clients cross the split: +0.1521 AUC
  -> that gap is client memorisation. Every number below uses the grouped split.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### The table

Every row ranks the same 30,000 pages against the same label. Model scores are **pooled out-of-fold** —
each page scored by the fold-model that never saw its client. The baseline rule is unchanged from ML-07
(asserted against its committed receipt in Section 1).

| System | p@20 | p@50 | p@100 | p@200 | p@500 | ROC-AUC |
|---|---|---|---|---|---|---|
| Flag everything (floor) | 0.542 | 0.542 | 0.542 | 0.542 | 0.542 | 0.500 |
| ML-07 rule baseline | 0.750 | 0.740 | 0.800 | **0.835** | 0.808 | 0.6485 |
| Decision tree (depth 3) | 0.600 | 0.600 | 0.570 | 0.600 | 0.600 | 0.6244 |
| Random forest | 0.550 | 0.680 | 0.740 | 0.780 | 0.760 | 0.6808 |
| Histogram gradient boosting | 0.900 | 0.820 | 0.790 | 0.800 | **0.812** | **0.6910** |
| Logistic regression | 0.800 | 0.880 | 0.810 | 0.805 | 0.782 | 0.6774 |
| **Hybrid — rule band, logistic orders within it** | **0.950** | **0.900** | **0.850** | 0.805 | 0.770 | 0.6606 |

Base rate **0.542**. **The hybrid takes precision@50 from 0.740 to 0.900** — from 37 of 50 slots landing
on a genuinely declining page to **45 of 50**, against 27 for random triage.

### The surprise: the readable model won

I expected boosting to win and wrote the candidate list in order of readability so that a simple winner
would be obvious if it happened. It happened. **Logistic regression beats the random forest and the
gradient booster at precision@50** (0.880 vs 0.680 and 0.820), and it does so *only because the count
columns were log-transformed* — in an earlier run without that step it scored 0.660 and finished last.
The honest lesson is not "linear models are underrated"; it is that **the ML-05 feature-engineering
decision was doing more work than the choice of model**, and I would have mis-attributed the result if I
had skipped it for the trees' sake.

The depth-3 tree (0.600) finishes **below the hand-written rule** — the one candidate that is more
readable than logistic regression is also the only one that cannot beat five human conditions.

### The finding that matters: the shape of the curve

ML-07 diagnosed a specific defect: *"precision RISES with K — the band is good, the ordering inside it is
not."* Re-measured here on the same rows:

| K | ML-07 rule | Boosting | Logistic | **Hybrid** |
|---|---|---|---|---|
| 10 | 0.700 | 1.000 | 0.700 | **1.000** |
| 20 | 0.750 | 0.900 | 0.800 | **0.950** |
| 50 | 0.740 | 0.820 | **0.880** | **0.900** |
| 100 | 0.800 | 0.790 | 0.810 | **0.850** |
| 200 | **0.835** | 0.800 | 0.805 | 0.805 |
| 500 | 0.808 | **0.812** | 0.782 | 0.770 |

**The rule's curve climbs; the hybrid's falls monotonically from 1.000.** A ranking that gets better the
deeper you read is mis-ordered at the top — which is fatal when only the first 20–50 rows are ever
opened. Plain logistic regression does not fully fix it either (0.700 at K=10, peaking at K=50). Only the
hybrid is monotone across every K I measured. **That is the result I would defend, more than the headline
0.740 → 0.900.**

Note the flip side, stated because it is equally true: **the rule still wins at K=200** (0.835) and
boosting wins at K=500. If capacity were 200 pages a sprint rather than 50, my recommendation would
change. The metric is only right because the capacity is what it is.

### How much of this is noise? Enough to be careful

precision@50 is a statement about **50 rows**; one row moves it by 0.02. Bootstrapping the selected set:

| System | p@50 | 95% interval | p@200 | 95% interval |
|---|---|---|---|---|
| ML-07 rule | 0.740 | [0.620, 0.860] | 0.835 | [0.785, 0.885] |
| Logistic regression | 0.880 | [0.780, 0.960] | 0.805 | [0.750, 0.860] |
| Hybrid | 0.900 | [0.820, 0.980] | 0.805 | [0.745, 0.860] |

The hybrid's interval [0.820, 0.980] and the rule's [0.620, 0.860] **overlap only slightly** — better
than the boosting-vs-rule comparison I ran first, where they overlapped heavily — but this is still one
dataset and one 90-day window. I would report the improvement as **measured and directional, not
established**.

### The spread across folds, which argues the other way

precision@50 measured *separately inside* each client-held-out fold:

| System | f1 | f2 | f3 | f4 | f5 | mean | **worst fold** |
|---|---|---|---|---|---|---|---|
| Decision tree (d3) | 0.500 | 0.600 | 0.580 | 0.720 | 0.700 | 0.620 | 0.500 |
| Logistic regression | 0.820 | 0.840 | 0.560 | 0.740 | 0.880 | 0.768 | **0.560** |
| Random forest | 0.780 | 0.920 | 0.840 | 0.660 | 0.660 | 0.772 | 0.660 |
| ML-07 rule | 0.720 | 0.780 | 0.780 | 0.960 | 0.680 | 0.784 | 0.680 |
| **Hybrid** | 0.840 | 0.920 | 0.740 | 0.840 | 0.760 | 0.820 | **0.740** |
| Histogram gradient boosting | 0.920 | 0.780 | 0.780 | 0.860 | 0.800 | **0.828** | **0.780** |

**Two different questions, two different winners, and I am not going to hide it.** Pooled out-of-fold
asks *"rank the whole portfolio at once"* and the hybrid wins. Per-fold asks *"rank a group of clients
this model has never seen"* and boosting wins on both mean and floor, while **plain logistic regression
collapses to 0.560 on fold 3** — barely above the base rate.

The gap is a calibration artifact worth naming: pooling requires scores to be comparable *across* fold
models, and a logistic regression's probabilities are far more comparable across fits than a tree
ensemble's. So logistic's pooled number partly reflects that its scores stack cleanly, not only that it
ranks well. **The hybrid is the compromise I actually trust**: second-best on both the pooled metric and
the fold floor, and the band structure is what stops any single fold model's mis-calibration from
dragging a page into the top 50.

### Reported because it was run, not because it flattered me

- **The random forest — the reference pipeline's model — lost at K=50** (0.680 vs the rule's 0.740)
  despite a better AUC (0.681 vs 0.649). Ranking quality and AUC are not the same claim.
- **The hybrid's AUC (0.6606) is the second-lowest in the table**, below plain logistic (0.6774) and
  boosting (0.6910). Gating on the rule's band throws away ordering information *below* the band, which
  AUC penalises and precision@50 never sees. I am optimising the top of a queue, so I accept that trade —
  but a reader who cares about the whole ranking should prefer boosting, and the table lets them see it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
K_VALUES = [20, 50, 100, 200, 500]

oof = {name: np.full(len(df), np.nan) for name in MODELS}     # pooled out-of-fold scores
per_fold = {name: [] for name in MODELS} | {"baseline_rule": [], "hybrid": []}
fitted = {name: [] for name in MODELS}

LINEAR = "logistic_regression"          # the model the hybrid orders with


def hybrid_key(band, model_score):
    """Rule band first (dominant), model probability orders inside it. Reason codes survive."""
    return band * 1000 + model_score * 100


for i, (tr, te) in enumerate(folds):
    per_fold["baseline_rule"].append(precision_at_k(y[te], baseline_rank_key[te], 50))
    for name in MODELS:
        pipe = make_pipe(name).fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        oof[name][te] = p
        per_fold[name].append(precision_at_k(y[te], p, 50))
        fitted[name].append((pipe, tr, te))
    per_fold["hybrid"].append(
        precision_at_k(y[te], hybrid_key(baseline_score[te], oof[LINEAR][te]), 50))
    print(f"fold {i+1}/5 trained")

assert not any(np.isnan(v).any() for v in oof.values()), "a row was never scored out-of-fold"
hybrid = hybrid_key(baseline_score, oof[LINEAR])

# --- the comparison table --------------------------------------------------
table = [("flag_everything_floor", {f"p@{k}": BASE_RATE for k in K_VALUES}, 0.5),
         ("baseline_rule", {f"p@{k}": precision_at_k(y, baseline_rank_key, k) for k in K_VALUES},
          roc_auc(y, baseline_score))]
table += [(n, {f"p@{k}": precision_at_k(y, oof[n], k) for k in K_VALUES}, roc_auc(y, oof[n])) for n in MODELS]
table += [("hybrid_rule_band_plus_logistic",
           {f"p@{k}": precision_at_k(y, hybrid, k) for k in K_VALUES}, roc_auc(y, hybrid))]

print("\n=== COMPARISON: same 30,000 rows, same metric, models scored out-of-fold ===")
print(f"{'system':<34}" + "".join(f"{'p@'+str(k):>9}" for k in K_VALUES) + f"{'ROC-AUC':>10}")
for name, m, auc in table:
    print(f"{name:<34}" + "".join(f"{m['p@'+str(k)]:>9.3f}" for k in K_VALUES) + f"{auc:>10.4f}")
print(f"{'(base rate)':<34}{BASE_RATE:>9.3f}")

ship = "hybrid_rule_band_plus_logistic"
best_model = max(MODELS, key=lambda n: precision_at_k(y, oof[n], 50))
print(f"\nbest single model by p@50: {best_model} | shipped ranking: {ship}")
print(f"slots landing on a declining page at K=50: "
      f"{precision_at_k(y, hybrid, 50) * 50:.0f}/50 (hybrid) vs "
      f"{precision_at_k(y, baseline_rank_key, 50) * 50:.0f}/50 (rule) vs "
      f"{BASE_RATE * 50:.0f}/50 (random triage)")

# --- the curve shape: the ML-07 defect, re-measured ------------------------
print("\nPrecision curve shape (ML-07's defect was 'precision RISES with K'):")
curves = {"ML-07 rule": baseline_rank_key, "boosting": oof["hist_gradient_boost"],
          "logistic": oof[LINEAR], "hybrid": hybrid}
print(f"  {'K':>5}" + "".join(f"{n:>14}" for n in curves))
for k in [10] + K_VALUES:
    print(f"  {k:>5}" + "".join(f"{precision_at_k(y, s, k):>14.3f}" for s in curves.values()))
for n, s in curves.items():
    vals = [precision_at_k(y, s, k) for k in [10] + K_VALUES]
    print(f"  {n:<12} monotone decreasing: {all(a >= b for a, b in zip(vals, vals[1:]))}")
print("  -> only the hybrid orders the very top correctly, which is where the queue is read")

# --- how much is noise? precision@50 is 50 rows ---------------------------
rng = np.random.default_rng(RANDOM_STATE)
def boot_ci(scores, k, n_boot=2000):
    picked = y[np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:k]]
    draws = picked[rng.integers(0, k, (n_boot, k))].mean(axis=1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

print("\n95% bootstrap interval on the selected set (one row of 50 moves p@50 by 0.02):")
for label, sc in [("ML-07 rule", baseline_rank_key), ("logistic", oof[LINEAR]), ("hybrid", hybrid)]:
    for k in (50, 200):
        lo, hi = boot_ci(sc, k)
        print(f"  {label:<12} p@{k:<4} = {precision_at_k(y, sc, k):.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
print("  -> hybrid vs rule barely overlap at K=50: measured and directional, not established")

# --- spread across folds: the floor, not the average ----------------------
print("\nprecision@50 inside each client-held-out fold (the floor is the operational number):")
print(f"  {'system':<24}" + "".join(f"{'f'+str(i+1):>7}" for i in range(5)) + f"{'mean':>8}{'worst':>8}")
for name, vals in sorted(per_fold.items(), key=lambda kv: np.mean(kv[1])):
    v = np.array(vals)
    print(f"  {name:<24}" + "".join(f"{x:>7.3f}" for x in v) + f"{v.mean():>8.3f}{v.min():>8.3f}")
print("  -> pooled ranking favours the hybrid; per-fold favours boosting. Both are reported.")

fold 1/5 trained


fold 2/5 trained


fold 3/5 trained


fold 4/5 trained


fold 5/5 trained



=== COMPARISON: same 30,000 rows, same metric, models scored out-of-fold ===
system                                 p@20     p@50    p@100    p@200    p@500   ROC-AUC
flag_everything_floor                 0.542    0.542    0.542    0.542    0.542    0.5000
baseline_rule                         0.750    0.740    0.800    0.835    0.808    0.6485
logistic_regression                   0.800    0.880    0.810    0.805    0.782    0.6774
decision_tree_d3                      0.600    0.600    0.570    0.600    0.600    0.6244
random_forest                         0.550    0.680    0.740    0.780    0.760    0.6808
hist_gradient_boost                   0.900    0.820    0.790    0.800    0.812    0.6910
hybrid_rule_band_plus_logistic        0.950    0.900    0.850    0.805    0.770    0.6606
(base rate)                           0.542

best single model by p@50: logistic_regression | shipped ranking: hybrid_rule_band_plus_logistic
slots landing on a declining page at K=50: 45/50 (hybrid) vs

    500         0.808         0.812         0.782         0.770
  ML-07 rule   monotone decreasing: False
  boosting     monotone decreasing: False
  logistic     monotone decreasing: False
  hybrid       monotone decreasing: True
  -> only the hybrid orders the very top correctly, which is where the queue is read

95% bootstrap interval on the selected set (one row of 50 moves p@50 by 0.02):


  ML-07 rule   p@50   = 0.740   95% CI [0.620, 0.860]
  ML-07 rule   p@200  = 0.835   95% CI [0.785, 0.885]
  logistic     p@50   = 0.880   95% CI [0.780, 0.960]
  logistic     p@200  = 0.805   95% CI [0.750, 0.860]
  hybrid       p@50   = 0.900   95% CI [0.820, 0.980]
  hybrid       p@200  = 0.805   95% CI [0.745, 0.860]
  -> hybrid vs rule barely overlap at K=50: measured and directional, not established

precision@50 inside each client-held-out fold (the floor is the operational number):
  system                       f1     f2     f3     f4     f5    mean   worst
  decision_tree_d3          0.500  0.600  0.580  0.720  0.700   0.620   0.500
  logistic_regression       0.820  0.840  0.560  0.740  0.880   0.768   0.560
  random_forest             0.780  0.920  0.840  0.660  0.660   0.772   0.660
  baseline_rule             0.720  0.780  0.780  0.960  0.680   0.784   0.680
  hybrid                    0.840  0.920  0.740  0.840  0.760   0.820   0.740
  hist_gradient_boost       0.920  0

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### What it leans on

Permutation importance for the logistic model (ROC-AUC drop when a column is shuffled), averaged over the
five held-out folds:

| Feature | Mean AUC drop | Reading |
|---|---|---|
| `impressions_90d` (log) | **0.119** | Volume dominates, by 2.4× over the next feature. |
| `clicks_90d` (log) | 0.050 | |
| `sessions_90d`, `users_90d` (log) | 0.033, 0.032 | |
| `avg_position` | 0.028 | The one non-volume feature near the top. |

**This is a volume-and-exposure profile, not a decay profile — and that is a finding I have to report
against my own result.** The capstone report flagged this risk in advance, before any model was trained:
*"if my own run reproduces the reference pipeline's importance profile, the honest reading is that the
model partly learns how much measurable traffic a page has."* It did reproduce it. The model is
substantially ranking pages by how much measurable search activity they carry, and only secondarily by
anything that looks like decay.

**Nothing is suspiciously perfect, which is the leakage check.** A single feature carrying 0.119 AUC in a
0.677-AUC model is a strong-but-ordinary signal. The alarm would have been a feature approaching 1.0 —
the shape ML-05 measured for `trend_pct` (0.753 inverted) before excluding it.

**The coefficients are printed below and I am deliberately not interpreting them one at a time.**
`impressions_90d` carries +1.18 while `clicks_90d` carries −0.62 and `users_90d` −0.59, on standardized
inputs. Those columns are strongly collinear — ML-05 named this exact risk — so the fit has split one
signal across several columns, and **the sign of any single coefficient is not a statement about the
world.** Read as a group they say something coherent (*high impressions relative to clicks and sessions*
— an exposed page that is not converting its exposure) but that reading is a hypothesis about the fit,
not a measured claim, and I flag it as such rather than putting it in the paper as an insight.

**The readable version, printed.** The depth-3 tree splits on `days_with_impressions ≤ 4.5`, then
`content_age_days ≤ 284.5`, then `avg_position ≤ 28.35` — legible as *pages with real sustained coverage,
of middling age, at a contestable position*. That is essentially the ML-07 rule rediscovered from the
data, which is both reassuring about the rule and an explanation of why the tree adds nothing over it.

### Where it is wrong

**1. The top-50 misses are growing pages — the baseline's failure mode, not a new one.** Of the 5 misses
in the hybrid's top 50, **4 measured `up`** (+23.6%, +71.3%, +84.8%, +93.7%) and 1 was `stable` (−8.7%).
ML-07's top-20 misses were the same species: pages growing +35% to +69%, put there by an impressions
tie-break. **Ranking accuracy improved; the failure mode did not change.**

**But the shared thread is position, not size** — and I had this wrong on a first reading. The five misses
span 84 to 1,593 impressions, so they are not "the biggest pages" the way ML-07's were. What they share is
**`avg_position` between 4.1 and 10.5** — every one is a page-1 page with sustained coverage. The model
cannot separate a *healthy* page-1 page from a *decaying* one, which is the same blind spot ML-06 measured
from the other direction: `top_3` pages decline least of any position band (0.241 against a 0.542 base
rate), so strong position is genuinely ambiguous evidence here. An editor sent to those 4 pages finds
healthy content and burns the slot.

**2. It cannot rank `comparison article` at all.** Out-of-fold AUC by content type: `feedly article`
**0.843** (n=2,096), `keyword article` **0.660** (n=27,207), `comparison article` **0.524** (n=697) —
chance. And the feedly number needs its own caveat: ML-04 measured that 46.6% of feedly articles are
labelled `new` and *structurally cannot* be `down`, so part of that 0.843 is the model identifying a
content type rather than detecting decay.

**3. It works far better for some clients than others.** Per-client out-of-fold AUC among clients with
≥200 pages runs from **0.506 to 0.770**, median **0.642**. Several clients get a ranking indistinguishable
from random. One client (476 pages, label rate 0.000) has no AUC at all — no positive cases exist, so the
metric is undefined, reported as such rather than filled in. A single portfolio-wide number hides all of
this, and a strategist working the 0.506 client would be right not to trust the queue.

**4. Concentration: the model helps, the hybrid hurts.** Top-50 client coverage — ML-07 rule: 8 clients,
largest supplying **44%**. Plain logistic: 8 clients, largest **30%** (an improvement). **The hybrid: 7
clients, largest 29 of 50 slots = 58%** (worse than either). The band gate concentrates the queue onto
whichever clients have many max-score pages. So the ranking I am recommending is also the one with the
worst portfolio coverage, and it needs a per-client cap before anything ships. **That is a product
decision the metric cannot see, and it is ML-10's job — named here so it cannot be quietly dropped.**

### What I take forward

**Ship the hybrid, capped per client, with the rule's reason codes attached.** All 50 rows of its queue
sit at baseline score 9, meaning all five conditions fired on every one of them — so every recommendation
still arrives with the same auditable explanation an editor could already read in ML-07, and the model
only decides the order. That combination is why I prefer it to the stronger stand-alone learner: **on
this data, boosting ranks marginally more robustly across unseen clients, and the hybrid ranks better
where the decision actually happens while remaining explainable.** If the next release shows the fold
floor mattering more than the pooled top-50, that preference should flip, and I would rather write that
condition down now than defend the choice later.

**The claim, in one sentence.** These pages **resemble the pages measured as declining in this 90-day
window** — not a forecast that they will decline, and not evidence that refreshing them recovers traffic.

**Reproducibility.** `random_state = 42` on every split, model and bootstrap; `clone()` on every
estimator so no state leaks between folds. Numbers here come from scikit-learn 1.9.0 / pandas 3.0.1 /
numpy 2.4.0 (printed by the setup cell). Tree-ensemble precision@K can move a point or two across library
versions — the direction of the curve is the finding, not the third decimal.

**The receipt.** `work/outputs/model_metrics.json` is committed; every model number in my capstone report
traces back to it. If the report and the file disagree, the report is wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance
from pathlib import Path

# --- what it leans on ------------------------------------------------------
imp_folds = []
for pipe, tr, te in fitted[LINEAR]:
    r = permutation_importance(pipe, X.iloc[te], y[te], n_repeats=3,
                               random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1)
    imp_folds.append(pd.Series(r.importances_mean, index=X.columns))
imp = pd.concat(imp_folds, axis=1)
imp.columns = [f"fold{i+1}" for i in range(len(imp_folds))]
imp["mean"] = imp.mean(axis=1)
imp = imp.sort_values("mean", ascending=False)
print(f"Permutation importance for {LINEAR}, averaged over the 5 held-out folds (AUC drop when shuffled):")
print(imp.head(8).round(4).to_string())
print(f"\n  top feature carries {imp['mean'].iloc[0]:.3f} AUC - strong but ordinary; a near-1.0 single "
      f"feature would be the leakage alarm (cf. trend_pct at 0.753 inverted, excluded in ML-05).")
print("  -> volume and exposure, not decay. Reported against my own result, as flagged in advance.")

# Coefficients: printed as a group, NOT interpreted one at a time (the count columns are collinear).
lin = make_pipe(LINEAR).fit(X, y)
coef = pd.Series(lin.named_steps["clf"].coef_[0],
                 index=lin.named_steps["pre"].get_feature_names_out()).sort_values()
print("\nLogistic coefficients (standardized inputs, fit on all rows - direction check only):")
print(pd.concat([coef.tail(6)[::-1], coef.head(6)]).round(3).to_string())
print("  -> impressions +, clicks/sessions/users - : ONE collinear signal split across columns.")
print("     Read as a group ('exposed but not converting'), never one coefficient at a time.")

# --- the readable version, printed ----------------------------------------
tr0, te0 = folds[1]
tree = make_pipe("decision_tree_d3").fit(X.iloc[tr0], y[tr0])
print("\nThe depth-3 tree (fit on fold 2's training clients) - the model you can read out loud:")
print(export_text(tree.named_steps["clf"],
                  feature_names=list(tree.named_steps["pre"].get_feature_names_out()), max_depth=3))

# --- where it is wrong -----------------------------------------------------
order = np.argsort(-hybrid, kind="stable")
top50 = df.iloc[order[:50]].copy()
print(f"SHIPPED ranking (hybrid) top 50: {int(top50['is_declining_label'].sum())}/50 genuinely declining "
      f"(random triage would give {BASE_RATE*50:.0f}/50)")
print(f"  baseline scores present in that top 50: {sorted(set(baseline_score[order[:50]]))} "
      f"-> all five reason codes fired on every row; the model only chose the order")

# top50 is a slice of df, so it already carries the label columns - no merge needed (a merge here
# would suffix them _x/_y and silently break the selection below).
misses = top50[top50["is_declining_label"] == 0]
print(f"\nThe {len(misses)} misses - what they ACTUALLY did (label columns read post-hoc, queue already frozen):")
print(misses[["impressions_90d", "days_with_impressions", "avg_position", "content_type",
              "trend_direction", "trend_pct"]].round(2).to_string(index=False))
print(f"  measured 'up': {(misses['trend_direction'] == 'up').sum()} of {len(misses)}"
      f" | avg_position range {misses['avg_position'].min():.1f}-{misses['avg_position'].max():.1f}"
      f" | impressions_90d range {misses['impressions_90d'].min():,}-{misses['impressions_90d'].max():,}")
print("  -> the common thread is POSITION, not size: every miss is a page-1 page with sustained")
print("     coverage that happens to be growing. The model cannot separate a healthy page-1 page")
print("     from a decaying one - the same blind spot ML-06 found (top_3 declines least, 0.241).")

print("\nOut-of-fold ROC-AUC by content_type (where the ranking has no signal at all):")
for t, g in df.groupby("content_type"):
    idx = g.index.to_numpy()
    print(f"  {t:<20} n={len(idx):>6,}  label_rate={y[idx].mean():.3f}  auc={roc_auc(y[idx], oof[LINEAR][idx]):.3f}")

print("\nOut-of-fold ROC-AUC by client (>=200 pages; identifiers withheld - public-safe):")
pc = pd.DataFrame([(len(g), y[g.index].mean(), roc_auc(y[g.index], oof[LINEAR][g.index]))
                   for _, g in df.groupby("client_id") if len(g) >= 200],
                  columns=["pages", "label_rate", "auc"]).sort_values("auc")
undefined = int(pc["auc"].isna().sum())
print(f"  {len(pc)} clients | AUC {pc['auc'].min():.3f} to {pc['auc'].max():.3f} | median {pc['auc'].median():.3f}"
      f" | undefined (single-class client): {undefined}")
print(pc.round(3).to_string(index=False, na_rep="n/a (no positives)"))

print("\nPortfolio coverage - the defect the metric cannot see:")
for label, sc in [("ML-07 rule", baseline_rank_key), ("logistic", oof[LINEAR]), ("hybrid (shipped)", hybrid)]:
    head = df.iloc[np.argsort(-np.asarray(sc, dtype=float), kind="stable")[:50]]
    counts = head["client_id"].value_counts()
    print(f"  {label:<18} top 50 spans {counts.size} clients | largest single client "
          f"{counts.iloc[0]}/50 ({counts.iloc[0]/50:.0%})")
print("  -> the shipped ranking has the WORST coverage. Needs a per-client cap before use (ML-10).")

# --- the receipt -----------------------------------------------------------
out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

queue = df.loc[order, ["content_id", "client_id", "content_type", "impressions_90d",
                       "days_with_impressions", "avg_position", "days_since_last_update",
                       "content_age_days", "is_declining_label"]].copy()
queue.insert(0, "queue_rank", np.arange(1, len(queue) + 1))
queue["hybrid_score"] = hybrid[order]
queue["model_score"] = oof[LINEAR][order]
queue["baseline_score"] = baseline_score[order]
queue.to_csv(out_dir / "model_action_score.csv", index=False)

metrics = {
    "base_rate": round(BASE_RATE, 4),
    "split": "GroupKFold(5) on client_id - zero client overlap, asserted",
    "evaluation": "pooled out-of-fold: every row scored by a fold-model blind to its client",
    "baseline_caveat": "the ML-07 rule's thresholds were read off all 30,000 rows, so the comparison "
                       "is tilted toward the baseline, not the model",
    "split_inflation_check": {"random_row_split_auc": 0.7765, "grouped_client_split_auc": 0.6244,
                              "note": "0.15 AUC of client memorisation - see Section 2"},
    "random_state": RANDOM_STATE,
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "shipped": ship,
    "best_single_model": best_model,
    "systems": {name: {**{k: round(v, 4) for k, v in m.items()}, "roc_auc": round(auc, 4)}
                for name, m, auc in table},
    "per_fold_precision_at_50": {n: [round(float(x), 4) for x in v] for n, v in per_fold.items()},
    "top_features_permutation_auc_drop": {k: round(float(v), 4) for k, v in imp["mean"].head(5).items()},
    "top50_client_concentration": {
        "distinct_clients": int(top50["client_id"].nunique()),
        "largest_client_share": round(float(top50["client_id"].value_counts().iloc[0] / 50), 4),
        "note": "worse than the ML-07 rule's 0.44 - a per-client cap is required (ML-10)",
    },
    "known_failure_mode": f"{int((misses['trend_direction'] == 'up').sum())} of {len(misses)} top-50 "
                          f"misses are growing pages - same failure mode as the ML-07 tie-break",
}
(out_dir / "model_metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f"\nwrote work/outputs/model_action_score.csv  ({len(queue):,} rows, gitignored)")
print("wrote work/outputs/model_metrics.json      (committed - the receipt for every model number)")

Permutation importance for logistic_regression, averaged over the 5 held-out folds (AUC drop when shuffled):
                    fold1   fold2   fold3   fold4   fold5    mean
impressions_90d    0.0575  0.0476  0.1461  0.1292  0.2123  0.1185
clicks_90d         0.0450  0.1344  0.0602  0.0169 -0.0049  0.0503
sessions_90d       0.0277  0.1230  0.0248 -0.0006 -0.0114  0.0327
users_90d          0.0147  0.1025  0.0530 -0.0039 -0.0049  0.0323
avg_position       0.0583  0.0263  0.0357 -0.0171  0.0375  0.0281
pageviews_90d      0.0101  0.0306  0.0045  0.0221  0.0725  0.0280
scroll_events_90d  0.0139  0.0130 -0.0014  0.0255  0.0409  0.0184
has_avg_position  -0.0001  0.0004  0.0529  0.0190  0.0125  0.0169

  top feature carries 0.119 AUC - strong but ordinary; a near-1.0 single feature would be the leakage alarm (cf. trend_pct at 0.753 inverted, excluded in ML-05).
  -> volume and exposure, not decay. Reported against my own result, as flagged in advance.



Logistic coefficients (standardized inputs, fit on all rows - direction check only):
num__impressions_90d                    1.181
num__has_avg_position                   0.795
num__pageviews_90d                      0.665
cat__impression_tier_low                0.523
num__scroll_events_90d                  0.429
cat__word_count_tier_1000-2000          0.385
num__clicks_90d                        -0.623
num__users_90d                         -0.592
cat__content_type_comparison article   -0.487
cat__freshness_tier_31-90              -0.486
num__sessions_90d                      -0.450
cat__impression_tier_excellent         -0.414
  -> impressions +, clicks/sessions/users - : ONE collinear signal split across columns.
     Read as a group ('exposed but not converting'), never one coefficient at a time.



The depth-3 tree (fit on fold 2's training clients) - the model you can read out loud:
|--- num__days_with_impressions <= 4.50
|   |--- num__has_avg_position <= 0.50
|   |   |--- num__scroll_rate <= 31.48
|   |   |   |--- class: 0
|   |   |--- num__scroll_rate >  31.48
|   |   |   |--- class: 0
|   |--- num__has_avg_position >  0.50
|   |   |--- num__days_with_impressions <= 2.50
|   |   |   |--- class: 0
|   |   |--- num__days_with_impressions >  2.50
|   |   |   |--- class: 0
|--- num__days_with_impressions >  4.50
|   |--- num__content_age_days <= 284.50
|   |   |--- num__days_with_impressions <= 25.50
|   |   |   |--- class: 0
|   |   |--- num__days_with_impressions >  25.50
|   |   |   |--- class: 1
|   |--- num__content_age_days >  284.50
|   |   |--- num__avg_position <= 28.35
|   |   |   |--- class: 1
|   |   |--- num__avg_position >  28.35
|   |   |   |--- class: 0

SHIPPED ranking (hybrid) top 50: 45/50 genuinely declining (random triage would give 27/50)
  baseline scores p


wrote work/outputs/model_action_score.csv  (30,000 rows, gitignored)
wrote work/outputs/model_metrics.json      (committed - the receipt for every model number)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.